# 01 Task 1: graph representation and traversal
**Problem.** Given a network $G=(V,E)$, build an adjacency-list and an adjacency-matrix representation, explore it with BFS and DFS, find its connected components, and measure how the representation affects time and memory.

**Input:** an instance (nodes, pipes). **Output:** both representations, traversal orders, the list of components, and measured memory and time for each representation.

**Technique: graph traversal.** BFS and DFS visit every vertex reachable from a start vertex exactly once, so repeating them from every unvisited vertex yields the connected components. They are the right tool because connectivity is a pure reachability question: no weights are involved.

In [1]:
import os, sys
if 'google.colab' in sys.modules and not os.path.isdir('../water'):
    os.system('git clone -q https://github.com/santihdzs/water-distribution')
    os.chdir('water-distribution/notebooks')
sys.path.insert(0, os.path.abspath('..'))

import inspect
import networkx as nx
import water
from water import graph, plot, timing

nets = water.load_all()

## Implementation
Adjacency list: dict id to list of `(neighbour, length)`. Adjacency matrix: $|V|\times|V|$ NumPy array with the pipe length (0 means no pipe) and an id-to-index map, needed because PES ids have gaps. BFS uses a FIFO queue, DFS an explicit stack. On the list, a vertex's neighbours are read directly; on the matrix, the whole row must be scanned.

In [2]:
print(inspect.getsource(graph.bfs_list))
print(inspect.getsource(graph.bfs_matrix))
print(inspect.getsource(graph.components))

def bfs_list(adj, s):
    parent = {s: None}
    order, queue = [], deque([s])
    while queue:
        u = queue.popleft()
        order.append(u)
        for v, _ in adj[u]:
            if v not in parent:
                parent[v] = u
                queue.append(v)
    return order, parent

def bfs_matrix(M, s):
    """s is a row index. scanning a full row per vertex makes this O(V^2)."""
    n = len(M)
    seen = [False] * n
    seen[s] = True
    order, queue = [], deque([s])
    while queue:
        u = queue.popleft()
        order.append(u)
        row = M[u]
        for v in range(n):
            if row[v] and not seen[v]:
                seen[v] = True
                queue.append(v)
    return order

def components(adj):
    """connected components as a list of sorted id lists, via repeated bfs."""
    seen, comps = set(), []
    for s in sorted(adj):
        if s in seen:
            continue
        order, _ = bfs_list(adj, s)
        seen.update(order)
        comps.appe

## Correctness
For every instance: both representations visit the same vertex set in BFS and DFS; BFS depth (pipes from the start) matches networkx; components match `nx.connected_components`. A disconnected case is also checked by removing all pipes at the highest-degree node of each instance.

In [3]:
def hop_levels(order, parent):
    lvl = {}
    for u in order:
        lvl[u] = 0 if parent[u] is None else lvl[parent[u]] + 1
    return lvl

for name, net in nets.items():
    adj = graph.adjacency_list(net)
    M, ids, index = graph.adjacency_matrix(net)
    G = nx.Graph(); G.add_nodes_from(net.coords); G.add_edges_from(p[:2] for p in net.pipes)
    s = net.sources[0]
    order, parent = graph.bfs_list(adj, s)
    order_m = [ids[i] for i in graph.bfs_matrix(M, index[s])]
    assert set(order) == set(order_m) == set(net.coords)
    assert set(graph.dfs_list(adj, s)) == {ids[i] for i in graph.dfs_matrix(M, index[s])} == set(net.coords)
    assert hop_levels(order, parent) == nx.single_source_shortest_path_length(G, s)
    comps = graph.components(adj)
    assert comps == [[ids[i] for i in c] for c in graph.components_matrix(M)]
    assert sorted(map(sorted, nx.connected_components(G))) == comps

    hub = max(adj, key=lambda u: (len(adj[u]), -u))
    cut = [p for p in net.pipes if hub not in p[:2]]
    k = len(graph.components(graph.adjacency_list(net, cut)))
    H = nx.Graph(); H.add_nodes_from(net.coords); H.add_edges_from(p[:2] for p in cut)
    assert k == nx.number_connected_components(H)
    depth = max(hop_levels(order, parent).values())
    print(f'{name}: connected ({len(comps)} component), BFS depth from source {s} = {depth} pipes; '
          f'removing the {len(adj[hub])} pipes at node {hub} leaves {k} components')

NYT: connected (1 component), BFS depth from source 17 = 9 pipes; removing the 4 pipes at node 9 leaves 3 components
HAN: connected (1 component), BFS depth from source 13 = 13 pipes; removing the 4 pipes at node 3 leaves 3 components
FOS: connected (1 component), BFS depth from source 7 = 7 pipes; removing the 4 pipes at node 1 leaves 3 components
PES: connected (1 component), BFS depth from source 1 = 20 pipes; removing the 5 pipes at node 55 leaves 2 components


All checks pass. Every instance is a single component, so every junction can be reached from at least one source. Removing the pipes around one hub splits each network into several components (the hub itself is one of them), which is the situation a sector isolation creates in task 3.

## Memory and time
Memory is the deep size of each structure (`water.timing.deep_size`: every referenced Python object counted once; for the matrix this includes the id-to-index map). Times are medians over 21 repeated samples (`water.timing.median_time`). The edge query asks whether a pipe joins two given nodes: $O(\deg u)$ on the list, $O(1)$ on the matrix.

In [4]:
rows = []
for name, net in nets.items():
    V, E = len(net.coords), len(net.pipes)
    adj = graph.adjacency_list(net)
    M, ids, index = graph.adjacency_matrix(net)
    mem_list = timing.deep_size(adj)
    mem_mat = timing.deep_size((M, ids, index))
    s = net.sources[0]
    t_bl = timing.median_time(lambda: graph.bfs_list(adj, s))
    t_bm = timing.median_time(lambda: graph.bfs_matrix(M, index[s]))
    pairs = [(u, v) for u in ids[:10] for v in ids[:10]]
    t_ql = timing.median_time(lambda: [any(w == v for w, _ in adj[u]) for u, v in pairs]) / len(pairs)
    t_qm = timing.median_time(lambda: [M[index[u], index[v]] > 0 for u, v in pairs]) / len(pairs)
    density = 2 * E / (V * (V - 1))
    rows.append([name, V, E, density, mem_list / 1024, mem_mat / 1024,
                 t_bl * 1e6, t_bm * 1e6, t_ql * 1e9, t_qm * 1e9])
    print(f'{name}: density {density:.3f} | memory list {mem_list/1024:.1f} KiB ({mem_list/(2*E):.0f} B per pipe end), matrix {mem_mat/1024:.1f} KiB | '
          f'BFS list {t_bl*1e6:.1f} us, matrix {t_bm*1e6:.1f} us ({t_bm/t_bl:.1f}x) | '
          f'edge query list {t_ql*1e9:.0f} ns, matrix {t_qm*1e9:.0f} ns')

plot.table('representation', rows,
           ['Inst.', '$|V|$', '$|E|$', 'Density', 'List (KiB)', 'Matrix (KiB)',
            r'BFS list ($\mu$s)', r'BFS matrix ($\mu$s)', 'Query list (ns)', 'Query matrix (ns)'],
           ['{}', '{}', '{}', '{:.3f}', '{:.1f}', '{:.1f}', '{:.1f}', '{:.1f}', '{:.0f}', '{:.0f}'])

NYT: density 0.111 | memory list 5.7 KiB (138 B per pipe end), matrix 4.7 KiB | BFS list 1.9 us, matrix 15.4 us (8.2x) | edge query list 128 ns, matrix 66 ns


HAN: density 0.069 | memory list 9.3 KiB (140 B per pipe end), matrix 10.5 KiB | BFS list 2.9 us, matrix 35.4 us (12.1x) | edge query list 132 ns, matrix 67 ns


FOS: density 0.087 | memory list 13.0 KiB (115 B per pipe end), matrix 13.4 KiB | BFS list 4.0 us, matrix 47.3 us (11.9x) | edge query list 149 ns, matrix 67 ns


PES: density 0.039 | memory list 23.3 KiB (122 B per pipe end), matrix 44.6 KiB | BFS list 7.3 us, matrix 173.2 us (23.9x) | edge query list 136 ns, matrix 67 ns


## Complexity
| | adjacency list | adjacency matrix |
|---|---|---|
| space | $O(V+E)$ | $O(V^2)$ |
| build | $O(V+E)$ | $O(V^2)$ (allocation) $+\,O(E)$ |
| BFS / DFS / components | $O(V+E)$ | $O(V^2)$ |
| is $(u,v)$ a pipe? | $O(\deg u)$ | $O(1)$ |

## Interpretation
The networks are very sparse: $E$ is between $1.05V$ and $1.57V$ and the density is at most 0.111, so a matrix spends almost all of its $V^2$ cells on pipes that do not exist. Memory is not a clean win for the list at this scale: each list entry is a Python tuple plus a float (about 120 bytes per pipe end) against 8 bytes per matrix cell, so on NYT the list is slightly larger. The matrix grows quadratically while the list grows linearly, and on PES the matrix is already about twice the size. BFS on the matrix is 8 to more than 20 times slower and the gap widens with $V$ because each visited vertex scans a full row; the list does work proportional to the pipes that exist. The matrix only wins on the pipe-existence query, which none of the tasks needs in an inner loop except Floyd-Warshall (task 6), which reads every pair anyway. The list is therefore the working representation for traversal, Dijkstra and flow; the matrix is used where an all-pairs table is the goal. Growth across sizes is analysed in `08_performance`.